In [1]:
import torch
import cvxpy as cp
import numpy as np
import pandas as pd
from IPython.display import display

def knapsack_specialized_pruning_sparse_leonardo(xi, v, w, C, device, delta):
    xi = xi.to(dtype=torch.float32, device=device)
    v = v.to(dtype=torch.float32, device=device)
    w = w.to(dtype=torch.float32, device=device)

    xi = xi - delta
    M = w.shape[0]

    b_list = []
    b = 0
    while True:
        delta_xi = xi[b + 1:] - xi[b]
        delta_v = v[b + 1:] - v[b]
        b = torch.argmin(delta_xi / delta_v) + 1 + b_list[-1] if b_list else 0
        if b != C - 1:
            b_list.append(int(b))
        if b + 1 > C - 1:
            break
    b_list.append(C - 1)

    x_plus = torch.zeros(C, dtype=torch.int32, device=device)
    x_plus[torch.tensor(b_list, device=device, dtype=torch.long)] = 1

    ratio = xi / v
    neg_indices = torch.where(v < 0)[0]
    pos_indices = torch.where(v >= 0)[0]
    neg_sorted = neg_indices[torch.argsort(ratio[neg_indices], descending=True)]
    pos_sorted = pos_indices[torch.argsort(ratio[pos_indices])]
    b_vector = torch.cat([neg_sorted, pos_sorted], dim=0).to(device)

    mask_small = w < v[0]
    mask_large = w > v[-1]
    mask_mid = (~mask_small) & (~mask_large)
    mask_edge = mask_small | mask_large

    idx_left = torch.zeros(M, dtype=torch.long, device=device)
    idx_right = torch.zeros(M, dtype=torch.long, device=device)
    theta = torch.zeros(M, dtype=torch.float32, device=device)

    if mask_edge.any():
        edge_idx = torch.nonzero(mask_edge, as_tuple=True)[0]
        w_edge = w[mask_edge]

        w_div_v0 = w_edge / v[0]
        w_div_v_last = w_edge / v[-1]

        edge_small = w_edge < v[0]
        edge_large = w_edge > v[-1]

        mask_cond_small = (w_div_v0 >= 0) & (w_div_v0 <= 1) & edge_small
        mask_else_small = edge_small & (~mask_cond_small)

        if mask_cond_small.any():
            inds = edge_idx[mask_cond_small]
            w_small = w_edge[mask_cond_small].unsqueeze(1)
            div_mat = w_small / v.unsqueeze(0)
            val_mat = div_mat * xi.unsqueeze(0)
            i_min = torch.argmin(val_mat, dim=1)
            vals_min = div_mat[torch.arange(i_min.shape[0], device=device), i_min]

            idx_left[inds] = i_min
            idx_right[inds] = i_min
            theta[inds] = vals_min

        if mask_else_small.any():
            inds = edge_idx[mask_else_small]
            idx_left[inds] = 0
            idx_right[inds] = 0
            theta[inds] = 1.0

        mask_cond_large = (w_div_v_last >= 0) & (w_div_v_last <= 1) & edge_large
        mask_else_large = edge_large & (~mask_cond_large)

        if mask_cond_large.any():
            inds = edge_idx[mask_cond_large]
            w_large = w_edge[mask_cond_large].unsqueeze(1)
            div_mat = w_large / v.unsqueeze(0)
            val_mat = div_mat * xi.unsqueeze(0)
            i_min = torch.argmin(val_mat, dim=1)
            vals_min = div_mat[torch.arange(i_min.shape[0], device=device), i_min]

            idx_left[inds] = i_min
            idx_right[inds] = i_min
            theta[inds] = vals_min

        if mask_else_large.any():
            inds = edge_idx[mask_else_large]
            idx_left[inds] = C - 1
            idx_right[inds] = C - 1
            theta[inds] = 1.0

    if mask_mid.any():
        mid_idx = torch.nonzero(mask_mid, as_tuple=True)[0]
        w_mid = w[mask_mid]

        ratio_b = w_mid[:, None] / v[b_vector]
        valid = (ratio_b >= 0) & (ratio_b <= 1) & (x_plus[b_vector] == 1).unsqueeze(0)

        valid_i0 = torch.where(
            valid,
            torch.arange(C, device=device)[None, :],
            torch.tensor(float("inf"), device=device)
        )

        i0_pos = valid_i0.argmin(dim=1)
        i0 = b_vector[i0_pos]
        v_i0 = v[i0]

        theta1 = w_mid / v_i0
        obj1 = theta1 * xi[i0]
        obj1 = torch.where(theta1 < 0, torch.tensor(float("inf"), device=device), obj1)

        one_indices = torch.nonzero(x_plus, as_tuple=True)[0].to(device=device, dtype=torch.long)
        i_right = torch.searchsorted(v[one_indices], w_mid, right=False)
        i_right = i_right.clamp(min=1, max=one_indices.shape[0] - 1)

        idx_r = one_indices[i_right]
        idx_l = one_indices[i_right - 1]

        v_left = v[idx_l]
        v_right = v[idx_r]

        theta2 = (w_mid - v_right) / (v_left - v_right + 1e-8)
        obj2 = theta2 * xi[idx_l] + (1.0 - theta2) * xi[idx_r]

        better_first = obj1 < obj2

        idx_left[mid_idx] = torch.where(better_first, i0, idx_l)
        idx_right[mid_idx] = torch.where(better_first, i0, idx_r)
        theta[mid_idx] = torch.where(better_first, theta1, theta2)

    denominator = v[idx_right] - v[idx_left]
    denominator_zero_mask = denominator == 0

    lambda_opt_nonzero = -(xi[idx_right] - xi[idx_left]) / denominator
    lambda_opt_zero_full = -xi / v
    lambda_opt_zero = lambda_opt_zero_full[idx_left]

    lambda_opt = torch.where(denominator_zero_mask, lambda_opt_zero, lambda_opt_nonzero)

    objective_values = delta + theta * xi[idx_left]

    mask_diff = idx_right != idx_left
    if mask_diff.any():
        objective_values[mask_diff] += (1.0 - theta[mask_diff]) * xi[idx_right[mask_diff]]

    x_placeholder = torch.stack(
        [idx_left.to(torch.int32), idx_right.to(torch.int32), theta],
        dim=1
    )

    return x_placeholder, lambda_opt, objective_values


def dense_from_leonardo_placeholder(x_placeholder, C, device):
    idx_left = x_placeholder[:, 0].to(dtype=torch.long, device=device)
    idx_right = x_placeholder[:, 1].to(dtype=torch.long, device=device)
    theta = x_placeholder[:, 2].to(dtype=torch.float32, device=device)

    M = x_placeholder.shape[0]
    rows = torch.arange(M, device=device)

    x_dense = torch.zeros(M, C, dtype=torch.float32, device=device)
    x_dense[rows, idx_left] += theta

    mask_diff = idx_right != idx_left
    x_dense[rows[mask_diff], idx_right[mask_diff]] += 1.0 - theta[mask_diff]

    return x_dense


def knapsack_solver_cvxpy(xi, v, w, C, delta):
    xi_np = xi.detach().cpu().numpy().astype(np.float64)
    v_np = v.detach().cpu().numpy().astype(np.float64)
    w_np = w.detach().cpu().numpy().astype(np.float64)
    delta_np = float(delta)

    M = w_np.shape[0]

    X_sol = np.zeros((M, C), dtype=np.float64)
    lambda_sol = np.zeros(M, dtype=np.float64)
    obj_sol = np.zeros(M, dtype=np.float64)

    installed = cp.installed_solvers()
    for candidate in ["CLARABEL", "ECOS", "OSQP", "SCS"]:
        if candidate in installed:
            solver = candidate
            break
    else:
        raise RuntimeError(f"Nessun solver CVXPY supportato trovato. Solver installati: {installed}")

    for i in range(M):
        x = cp.Variable(C)

        objective = cp.Minimize(xi_np @ x + delta_np * (1.0 - cp.sum(x)))

        constraints = [
            v_np @ x == w_np[i],
            cp.sum(x) <= 1.0,
            x >= 0.0,
            x <= 1.0,
        ]

        prob = cp.Problem(objective, constraints)

        if solver == "SCS":
            prob.solve(solver=solver, verbose=False, eps=1e-8, max_iters=200000)
        else:
            prob.solve(solver=solver, verbose=False)

        if prob.status not in ["optimal", "optimal_inaccurate"]:
            raise RuntimeError(
                f"CVXPY non ha trovato una soluzione per i={i}, "
                f"w={w_np[i]:.6f}, status={prob.status}"
            )

        X_sol[i] = np.where(np.abs(x.value) < 1e-8, 0.0, x.value)
        obj_sol[i] = prob.value

        lam = constraints[0].dual_value
        lambda_sol[i] = lam if lam is not None else np.nan

    return (
        torch.tensor(X_sol, dtype=torch.float32),
        torch.tensor(lambda_sol, dtype=torch.float32),
        torch.tensor(obj_sol, dtype=torch.float32),
        solver,
    )


def make_v_from_w_percentiles_with_bucket_zero(w, C, device):
    assert C >= 3, "Serve C >= 3."
    assert C % 2 == 1, "Serve C dispari per avere un valore centrale 0 esatto."

    w = w.to(dtype=torch.float32, device=device)

    p_low = torch.quantile(w, 0.001)
    p_high = torch.quantile(w, 0.999)

    assert p_low < 0 < p_high, (
        f"I percentili devono contenere 0: p_low={p_low.item()}, p_high={p_high.item()}"
    )

    mid = C // 2

    v_left = torch.linspace(p_low.item(), 0.0, steps=mid + 1, device=device)
    v_right = torch.linspace(0.0, p_high.item(), steps=mid + 1, device=device)[1:]

    v = torch.cat([v_left, v_right])
    v[mid] = 0.0

    return v.to(dtype=torch.float32), p_low, p_high


device = torch.device("cpu")

C = 5
M = 100
delta = -100
r_w_generation = 0.20

xi = torch.randn(C, device=device)
xi = torch.sort(xi)[0]

w = -r_w_generation + 2.0 * r_w_generation * torch.rand(M, device=device)

v, p_low, p_high = make_v_from_w_percentiles_with_bucket_zero(w, C, device)

w = torch.clamp(w, min=p_low.item(), max=p_high.item())
w = w[torch.randperm(M)]

print("=== INPUT ===")
print("xi    =", xi)
print("v     =", v)
print("p0.1  =", p_low)
print("p99.9 =", p_high)
print("w     =", w)
print("C     =", C)
print("delta =", delta)

x_sparse, lambda_leo, obj_leo_returned = knapsack_specialized_pruning_sparse_leonardo(
    xi.clone(), v.clone(), w.clone(), C, device, delta
)

x_leo_dense = dense_from_leonardo_placeholder(x_sparse, C, device)

tol = 1e-8

active_per_row = (torch.abs(x_leo_dense) > tol).sum(dim=1)

num_zero_active = (active_per_row == 0).sum().item()
num_one_active = (active_per_row == 1).sum().item()
num_two_active = (active_per_row == 2).sum().item()
num_more_than_two_active = (active_per_row > 2).sum().item()

print("\n=== SPARSITÀ x Leonardo ===")
print(f"righe con 0 componenti attive  = {num_zero_active}")
print(f"righe con 1 componente attiva  = {num_one_active}")
print(f"righe con 2 componenti attive  = {num_two_active}")
print(f"righe con >2 componenti attive = {num_more_than_two_active}")

obj_leo_recomputed = delta + x_leo_dense @ (xi - delta)

x_cvxpy, lambda_cvxpy, obj_cvxpy, solver_used = knapsack_solver_cvxpy(
    xi.clone(), v.clone(), w.clone(), C, delta
)

w_residual_leo = x_leo_dense @ v - w
sum_x_leo = x_leo_dense.sum(dim=1)

max_x_err = torch.max(torch.abs(x_leo_dense - x_cvxpy)).item()
max_obj_err_returned = torch.max(torch.abs(obj_leo_returned.cpu() - obj_cvxpy)).item()
max_obj_err_recomputed = torch.max(torch.abs(obj_leo_recomputed.cpu() - obj_cvxpy)).item()
max_feas_err = torch.max(torch.abs(w_residual_leo)).item()
max_sum_violation = torch.clamp(sum_x_leo - 1.0, min=0.0).max().item()

print("\n=== CONFRONTO ===")
print(f"Solver CVXPY usato: {solver_used}")
print(f"max |x_leonardo_dense - x_cvxpy|             = {max_x_err:.3e}")
print(f"max |obj_returned_leonardo - obj_cvxpy|      = {max_obj_err_returned:.3e}")
print(f"max |obj_recomputed_from_sparse - obj_cvxpy| = {max_obj_err_recomputed:.3e}")
print(f"max |v @ x_leonardo - w|                     = {max_feas_err:.3e}")
print(f"max violation(sum(x) <= 1)                   = {max_sum_violation:.3e}")

if max_obj_err_recomputed < 1e-4 and max_feas_err < 1e-4 and max_sum_violation < 1e-4:
    print("\nOK: la x sparsa ricostruita è coerente con CVXPY.")
else:
    print("\nATTENZIONE: la x sparsa ricostruita NON coincide numericamente con CVXPY.")

if max_obj_err_returned >= 1e-4 and max_obj_err_recomputed < 1e-4:
    print(
        "Nota: objective_values ritornato dalla funzione non coincide, "
        "ma l'obiettivo ricalcolato dalla x sparsa sì. "
        "Questo segnala un possibile problema nella formula di objective_values "
        "quando idx_left == idx_right."
    )

df = pd.DataFrame({
    "w": w.detach().cpu().numpy(),
    "idx_left": x_sparse[:, 0].detach().cpu().numpy().astype(int),
    "idx_right": x_sparse[:, 1].detach().cpu().numpy().astype(int),
    "theta": x_sparse[:, 2].detach().cpu().numpy(),
    "obj_cvxpy": obj_cvxpy.detach().cpu().numpy(),
    "obj_returned_leonardo": obj_leo_returned.detach().cpu().numpy(),
    "obj_recomputed_from_sparse": obj_leo_recomputed.detach().cpu().numpy(),
    "abs_err_obj_returned": torch.abs(obj_leo_returned.cpu() - obj_cvxpy).numpy(),
    "abs_err_obj_recomputed": torch.abs(obj_leo_recomputed.cpu() - obj_cvxpy).numpy(),
})

display(df)

print("\n=== x Leonardo densa ===")
print(x_leo_dense)

print("\n=== x CVXPY ===")
print(x_cvxpy)

=== INPUT ===
xi    = tensor([-1.3924, -0.0761,  0.2630,  0.3058,  0.3642])
v     = tensor([-0.1956, -0.0978,  0.0000,  0.0990,  0.1979])
p0.1  = tensor(-0.1956)
p99.9 = tensor(0.1979)
w     = tensor([ 0.0396, -0.1091,  0.1185,  0.1415,  0.1166,  0.1025,  0.0955, -0.1774,
         0.0196, -0.1453, -0.1043, -0.0599,  0.1547,  0.0365,  0.0692, -0.1382,
        -0.1036,  0.0455, -0.1512, -0.0982, -0.0922, -0.0032, -0.1430, -0.0770,
        -0.0893, -0.0624,  0.1902,  0.1394,  0.1551, -0.0657, -0.1080,  0.0780,
         0.0547,  0.1380,  0.1513,  0.0394,  0.0306,  0.1411,  0.1417,  0.0455,
        -0.0725, -0.0942,  0.1079,  0.0539,  0.1336, -0.1367, -0.1956,  0.0311,
        -0.0618,  0.0662,  0.1391, -0.1067, -0.1497, -0.0151,  0.0790,  0.0447,
        -0.1562,  0.1740,  0.0357,  0.1235, -0.1142,  0.0283,  0.1539,  0.1922,
         0.1025, -0.1710,  0.1284, -0.1324,  0.0905,  0.1944, -0.1473, -0.0308,
         0.1531,  0.1754,  0.1257, -0.1925,  0.0282,  0.0115, -0.1036, -0.1858,
       

,w,idx_left,idx_right,theta,obj_cvxpy,obj_returned_leonardo,obj_recomputed_from_sparse,abs_err_obj_returned,abs_err_obj_recomputed
0,0.039604,4,4,0.200088,-79.918373,-79.918365,-79.918365,0.000008,0.000008
1,-0.109137,0,0,0.557873,-44.989468,-44.989468,-44.989468,0.000000,0.000000
2,0.118543,4,4,0.598910,-39.890881,-39.890881,-39.890881,0.000000,0.000000
3,0.141516,4,4,0.714976,-28.241993,-28.241997,-28.241997,0.000004,0.000004
4,0.116629,4,4,0.589240,-40.861370,-40.861366,-40.861366,0.000004,0.000004
...,...,...,...,...,...,...,...,...,...
95,-0.050649,0,0,0.258903,-74.470230,-74.470230,-74.470230,0.000000,0.000000
96,-0.139754,0,0,0.714381,-29.556635,-29.556633,-29.556633,0.000002,0.000002
97,-0.123941,0,0,0.633548,-37.527340,-37.527340,-37.527340,0.000000,0.000000
98,-0.101532,0,0,0.519001,-48.822529,-48.822529,-48.822529,0.000000,0.000000



=== x Leonardo densa ===
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.2001],
        [0.5579, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5989],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7150],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5892],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5178],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.4823],
        [0.9066, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0991],
        [0.7430, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5332, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3060, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7818],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.1845],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3494],
        [0.7063, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5295, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.2296],
        [0.7731, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5022, 0.0000,